# 第十四课｜什么是现场可编程门阵列（Field-Programmable Gate Array, FPGA）开发板？

上一课得到的是**配置比特流（bitstream）**。今天第一次把视线从“芯片内部逻辑”移到真实开发板：

> **一颗 FPGA 芯片为什么还需要电源、时钟、reset、输入/输出和外设才能成为实验平台？**

主要新概念：**开发板是围绕 FPGA 芯片构建的完整硬件环境。**

## 1. 概念账本

**已经知道：** FPGA 可以被 bitstream 配置；数字状态依赖 clock。

**今天学习：**
- **开发板（development board）**；
- **输入/输出（Input/Output, I/O）**；
- board 上的 clock source、reset、power、connector/peripheral。

**只预告：** 后续再学 host↔FPGA 通信、外部内存和标准片上通信。

## 2. FPGA 芯片不等于 FPGA 开发板

<svg xmlns="http://www.w3.org/2000/svg" width="760" height="300" viewBox="0 0 760 300" role="img" aria-label="fpga development board diagram">
  <defs>
    <marker id="arrow-board" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <path d="M0,0 L8,4 L0,8 Z" fill="#506070"/>
    </marker>
  </defs>
  <rect x="270" y="82" width="220" height="136" rx="18" fill="#e6f4ea" stroke="#506070" stroke-width="2"/>
  <text x="380" y="142" text-anchor="middle" font-size="22" font-family="sans-serif">FPGA chip</text>
  <text x="380" y="172" text-anchor="middle" font-size="14" font-family="sans-serif">programmable logic</text>
  <rect x="30" y="24" width="150" height="42" rx="10" fill="#e6f4ea" stroke="#506070"/>
  <text x="105" y="50" text-anchor="middle" font-size="15" font-family="sans-serif">power</text>
  <line x1="180" y1="45" x2="270" y2="102" stroke="#506070" stroke-width="2" marker-end="url(#arrow-board)"/>
  <rect x="30" y="94" width="150" height="42" rx="10" fill="#e6f4ea" stroke="#506070"/>
  <text x="105" y="120" text-anchor="middle" font-size="15" font-family="sans-serif">clock source</text>
  <line x1="180" y1="115" x2="270" y2="132" stroke="#506070" stroke-width="2" marker-end="url(#arrow-board)"/>
  <rect x="30" y="164" width="150" height="42" rx="10" fill="#e6f4ea" stroke="#506070"/>
  <text x="105" y="190" text-anchor="middle" font-size="15" font-family="sans-serif">reset</text>
  <line x1="180" y1="185" x2="270" y2="170" stroke="#506070" stroke-width="2" marker-end="url(#arrow-board)"/>
  <rect x="30" y="234" width="150" height="42" rx="10" fill="#e6f4ea" stroke="#506070"/>
  <text x="105" y="260" text-anchor="middle" font-size="15" font-family="sans-serif">configuration path</text>
  <line x1="180" y1="255" x2="270" y2="202" stroke="#506070" stroke-width="2" marker-end="url(#arrow-board)"/>
  <rect x="570" y="116" width="160" height="68" rx="10" fill="#e6f4ea" stroke="#506070"/>
  <text x="650" y="143" text-anchor="middle" font-size="15" font-family="sans-serif">I/O connectors</text>
  <text x="650" y="165" text-anchor="middle" font-size="15" font-family="sans-serif">/ peripherals</text>
  <line x1="490" y1="143" x2="570" y2="143" stroke="#506070" stroke-width="2" marker-end="url(#arrow-board)"/>
  <line x1="570" y1="164" x2="490" y2="164" stroke="#506070" stroke-width="2" marker-end="url(#arrow-board)"/>
</svg>

同一类 FPGA 可以出现在不同开发板上，因此课程先讲通用结构，再把板卡专用步骤留到真正部署时。

## 3. clock 与肉眼可见之间差很多

板上 clock 常常每秒运行数千万到数亿个 cycle。LED、人手按钮和终端输出慢得多。

所以第一个物理实验通常不是神经网络，而是 counter：用高速 clock 计数，再把一个慢得多的变化暴露到 LED 或可读 register。

## 4. Run：把 100 MHz 变成可观察节拍

假设输入 clock 是 100 MHz，希望每秒产生 2 次可观察 tick。先预测需要多少 cycle。

In [ ]:
clock_hz = 100_000_000
target_tick_hz = 2

cycles_per_tick = clock_hz // target_tick_hz
counter_bits = max(1, (cycles_per_tick - 1).bit_length())

print("cycles per visible tick:", cycles_per_tick)
print("counter bits required:", counter_bits)


## 5. Observe

每 50,000,000 个 cycle 产生一个 tick，只需约 26 bit 就能表示 `0..49,999,999`。慢速 LED 背后仍然是高速同步电路。

## 6. reset 在这里做什么？

reset 不是“把整块板恢复出厂设置”。在教学电路里，它通常让指定 register/state 回到已知初值。

第一次物理 proof 只需要上电、clock、reset 和一个可观察输出，不应同时引入 DDR、host transport 与神经网络。

## 7. 板卡选择的项目边界

RMD 当前把 **KV260 级开发板**作为默认候选，但要求平台接口可替换。课程不会把“会用某一块板”当成“理解 FPGA”的同义词。

下一课再解释这类平台为什么常把处理器系统与可编程逻辑放在一起。

## 8. Try It

把 `target_tick_hz` 改成 10。先预测 `cycles_per_tick` 与 `counter_bits` 怎样变化。

本例假设输入 clock 能被目标 tick 频率整除。

## 9. 作业

[第 14 课作业：把板上时钟变成可观察 tick](../../exercises/zh/14_what_is_fpga_board.ipynb)

正式作业不需要购买开发板。

## 10. AI Task

让 AI 根据候选开发板资料列出核心 FPGA/可编程逻辑器件、clock、reset、用户 I/O、host connection、external memory 六类资源；检查它是否把板级资源和 FPGA 内部逻辑混为一谈。

## 11. Human Check

指出 FPGA 芯片与开发板的区别；为什么第一个板上实验常用 counter/LED；clock source、reset、I/O 各自做什么；为什么本课在无板情况下仍能完成。

## 12. Engineering Handoff

对应 `RMD-012 / RMD-012A`：选择板卡与 platform shell 后，用 counter / LED 或可观察 register 做 first physical proof。对应的 KV260 Physical Lab 是正式实体实操路径；本课概念与作业仍可在无板时先完成。

## 13. 项目追踪 Project Trace

- Lesson: `LSN-014`
- Mapping: `RMD-012 / RMD-012A`
- Physical proof: clock + reset + observable state
- Platform policy: replaceable board-specific shell

## 14. Exit Ticket

你能把 FPGA 开发板拆成“芯片 + 供电/clock/reset + I/O/外设 + 配置路径”，并解释为什么第一项物理验证应该很小。